# 07 · matplotlib 制图（进阶）

把基础绘图延伸到立体空间、色彩系统、复杂版面、动画与图片内嵌输出，做出能对外发布的进阶图表。

## 学习目标
- 能用 3D 座标轴（3D axes）画出立体散点、立体长条与曲面，并调整观看视角。
- 理解色彩映射（colormap）与范式（normalization）的关系，能让多张图共用同一条色阶。
- 能设计多子图（subplots）网格版面，善用轴共享（sharex / sharey）与循环批量填图。
- 能用动画（animation）产生逐格画面并输出成 GIF 档。
- 能把图表转成内存字节流（in-memory bytes），编码成 base64 内嵌到 HTML，并用图像库做缩略图。

In [ ]:
# 中文字体设置（本书会画图才需要）
import os
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for fp in ['NotoSansCJK-Regular.otf', 'msjh.ttc', 'mingliu.ttc']:
    try:
        if os.path.exists(fp):
            fm.fontManager.addfont(fp)
    except Exception:
        pass
plt.rcParams['font.family'] = ['Microsoft JhengHei', 'Noto Sans CJK TC', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

## 3D 绘图基础（3D plotting）

3D 绘图是在带有第三个维度（z 轴）的座标轴上作图，比 2D 多了深度与观看视角（view angle）。
当数据本身有三个量（例如平面位置加高度），用立体图能一眼看出空间分布。

与 2D 的关键差异：
- 轴对象不同：用 `add_subplot(projection="3d")` 取得 3D 轴（Axes3D）。
- 多了视角控制：`view_init(elev, azim)` 设置仰角（elevation）与方位角（azimuth）。

形状：
```
ax = fig.add_subplot(projection="3d")
ax.scatter(x, y, z)
ax.view_init(elev=仰角, azim=方位角)
```

In [ ]:
# 概念：3D 散点与立体长条，并用两种视角输出对比
import numpy as np

rng = np.random.default_rng(0)        # 固定乱数种子，让示范可重现
x = rng.uniform(0, 10, 40)            # 自造 40 个点的平面 x 座标
y = rng.uniform(0, 10, 40)            # 自造平面 y 座标
z = rng.uniform(0, 5, 40)            # 自造高度当作 z 轴

fig = plt.figure(figsize=(10, 4))
for i, (elev, azim) in enumerate([(20, 30), (60, 120)]):   # 两组视角
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")      # 开立体轴
    ax.scatter(x, y, z, c=z, cmap="viridis")               # 依高度上色的 3D 散点
    ax.view_init(elev=elev, azim=azim)                     # 设置仰角与方位角
    ax.set_title(f"视角 elev={elev}, azim={azim}")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
plt.tight_layout()
plt.show()

# 概念：立体长条图 bar3d，需给每根长条的底座位置与长宽高
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(projection="3d")
gx, gy = np.meshgrid(range(3), range(3))      # 造 3x3 网格底座座标
gx = gx.ravel(); gy = gy.ravel()             # 摊平成一维，bar3d 要一维输入
heights = rng.uniform(1, 6, gx.size)          # 每根长条的高度（仿真数据）
# 注意：bar3d(x, y, z, dx, dy, dz)，z 是底座高度（这里都从 0 起算）
ax.bar3d(gx, gy, np.zeros_like(heights), 0.6, 0.6, heights, shade=True)
ax.set_title("立体长条图 bar3d")
plt.show()

## 色彩映射与范式（colormap & normalization）

色彩映射（colormap）是一张「数值到颜色」的对照表；数值要先经过范式（normalization）映到 0 到 1，才能查表取色。
搞懂这层关系，颜色才能正确表达数据大小，而不是随意上色。

两种对应方式：

| 方式 | 对象 | 适用 |
|---|---|---|
| 连续 | `Normalize` | 数值连续变化、看渐层 |
| 离散分级 | `BoundaryNorm` + `ListedColormap` | 想把数值切成几个级距、看分组 |

In [ ]:
# 概念：同一份数值，连续 colormap 对比离散 BoundaryNorm
from matplotlib.colors import Normalize, BoundaryNorm, ListedColormap

data = np.linspace(0, 100, 100).reshape(10, 10)   # 自造 0 到 100 的方阵当示范值

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

# 连续：Normalize 把数据线性压到 0..1，再查 viridis 色票
im0 = axes[0].imshow(data, cmap="viridis", norm=Normalize(vmin=0, vmax=100))
axes[0].set_title("连续 colormap")
fig.colorbar(im0, ax=axes[0])

# 离散：用边界把数值切成四级，每级一个固定颜色
bounds = [0, 25, 50, 75, 100]                       # 四个级距的边界
discrete = ListedColormap(["#ffffcc", "#a1dab4", "#41b6c4", "#225ea8"])
bnorm = BoundaryNorm(bounds, discrete.N)            # 依边界把值归到某一级
im1 = axes[1].imshow(data, cmap=discrete, norm=bnorm)
axes[1].set_title("离散 BoundaryNorm")
fig.colorbar(im1, ax=axes[1], ticks=bounds)
plt.tight_layout()
plt.show()
# 技巧：离散分级适合做「低中高」这种等级图，读者更容易看出群组
print("连续看渐层，离散看分组")

## 色条与共用色阶（colorbar & shared scale）

色条（colorbar）是把色彩映射画成一条图例，告诉读者颜色对应什么数值。
当多张图要互相比较时，必须共用同一个数值范围（shared color scale），否则同一颜色在不同图代表不同数值，会误导读者。

做法：创建一个只负责「值到色」的纯色彩映射对象 `ScalarMappable`，绑定统一的 `Normalize`，再用它画出一条共用色条。

In [ ]:
# 概念：先看各自色阶的误导，再用统一 Normalize + ScalarMappable 修正
from matplotlib.cm import ScalarMappable

# 自造三组数值范围差很多的小数据
groups = [rng.uniform(0, 10, (5, 5)),
          rng.uniform(0, 50, (5, 5)),
          rng.uniform(0, 100, (5, 5))]

# 各自色阶：每张自动依自己的最小最大值上色，颜色无法互相比较
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, g in zip(axes, groups):
    im = ax.imshow(g, cmap="viridis")   # 没给 norm，各自独立缩放
    fig.colorbar(im, ax=ax)
fig.suptitle("各自色阶：同色不同义（误导）")
plt.tight_layout(); plt.show()

# 修正：用全域最小最大值建一个共用 Normalize
vmin = min(g.min() for g in groups)
vmax = max(g.max() for g in groups)
shared = Normalize(vmin=vmin, vmax=vmax)            # 三张共用同一数值范围
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, g in zip(axes, groups):
    ax.imshow(g, cmap="viridis", norm=shared)       # 全部套同一 norm
sm = ScalarMappable(norm=shared, cmap="viridis")    # 纯色彩映射对象，当共用色条来源
fig.colorbar(sm, ax=axes, label="统一数值范围")       # 一条色条服务三格
fig.suptitle("共用色阶：颜色可比较")
plt.show()
print(f"全域范围 vmin={vmin:.1f}, vmax={vmax:.1f}")

## 多子图排版（subplots layout）

多子图（subplots）是在一张画布上排出多个面板的网格，常用于把多组数据并列比较。
用循环批量填图（loop filling）能系统化地把每组数据填进对应格子，避免一格一格手写。

轴共享（sharex / sharey）让同一排或同一列共用刻度，省去重复标示、让对比更清楚。

形状：
```
fig, axes = plt.subplots(列数, 行数, sharex=True)
for ax, 数据 in zip(axes.ravel(), 数据群):
    ax.plot(...)
```

In [ ]:
# 概念：用循环把多组同结构串行填进 2xN 网格，并共享 x 轴
t = np.linspace(0, 2 * np.pi, 100)      # 共用的时间轴
n = 3                                    # 每排三格
# 自造 6 组同结构串行：不同频率与振幅的正弦波
series = [("freq=%d" % f, a * np.sin(f * t)) for f, a in
          [(1, 1), (2, 1), (3, 1), (1, 2), (2, 2), (3, 2)]]

fig, axes = plt.subplots(2, n, figsize=(11, 5), sharex=True)   # 上下两排共享 x 轴
for ax, (label, ys) in zip(axes.ravel(), series):              # ravel 把 2D 轴数组摊平成一维好循环
    ax.plot(t, ys)
    ax.set_title(label)
    ax.axhline(0, color="gray", lw=0.5)
# 注意：sharex 之下只有最底排会显示 x 刻度，自动省去上排重复标示
fig.suptitle("2x3 子图网格（共享 x 轴）")
plt.tight_layout()
plt.show()
print(f"共填入 {len(series)} 个子图")

## 动画与 GIF 输出（animation & GIF）

动画（animation）是「同一张图反复更新内容」的逐格产物；每一格称为影格（frame）。
`FuncAnimation` 会对每个影格调用一个更新函数（update function），由它改变图上的数据。

理解更新函数如何随影格改变数据，就能把静态图变成动态 GIF，最后用 `PillowWriter` 写出 GIF 档。

In [ ]:
# 概念：FuncAnimation 逐格更新正弦波，PillowWriter 输出 GIF
from matplotlib.animation import FuncAnimation, PillowWriter

xs = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(6, 3))
line, = ax.plot(xs, np.sin(xs))         # 先画第 0 格，line 对象之后反复更新
ax.set_ylim(-1.1, 1.1)
ax.set_title("随时间推进的正弦波")

def update(frame):                       # 更新函数：frame 是目前影格编号
    line.set_ydata(np.sin(xs + frame * 0.2))   # 相位随影格平移，看起来像在动
    return (line,)                       # 回传被改动的对象

# 注意：frames 决定总影格数，interval 是每格毫秒数
anim = FuncAnimation(fig, update, frames=30, interval=80, blit=True)
out_gif = "wave.gif"
anim.save(out_gif, writer=PillowWriter(fps=12))   # 用 Pillow 写出 GIF
plt.close(fig)                            # 关掉避免在 notebook 多印一张静态图
print("已输出 GIF：", out_gif, "存在吗：", os.path.exists(out_gif))

## 图内嵌与图像处理（embed & image）

把图表不落地存盘，而是直接存进内存字节流（in-memory bytes），再做 base64 编码（base64 encoding）内嵌到 HTML，是报表与仪表板常见手法。
之后可用图像库 PIL 做缩略图（thumbnail）控制文件大小。

流程：
1. `savefig(BytesIO)` 把图存进内存缓冲。
2. `base64` 把字节编成可放进 HTML 的字符串。
3. 组成 `<img src="data:image/png;base64,...">` 标签。
4. 用 PIL 开缓冲、`thumbnail` 缩小尺寸。

In [ ]:
# 概念：图存进 BytesIO，编 base64 组 <img>，再用 PIL 缩略图
import base64
from io import BytesIO
from PIL import Image

# 自造一张折线图
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(np.arange(10), np.cumsum(rng.normal(size=10)))   # 累积随机漫步当示范线
ax.set_title("待内嵌的折线图")

buf = BytesIO()                           # 内存字节流，当虚拟文件
fig.savefig(buf, format="png", dpi=80)    # 把图写进缓冲而非磁盘
plt.close(fig)
buf.seek(0)                               # 写完后把读取位置移回开头
png_bytes = buf.getvalue()

b64 = base64.b64encode(png_bytes).decode("ascii")   # 字节编成纯文字
img_tag = f'<img src="data:image/png;base64,{b64}">'  # 可直接贴进 HTML 的标签
print("img 标签开头：", img_tag[:60], "...")

# 技巧：用 PIL 开同一份字节流做缩略图，控制输出大小
buf.seek(0)
im = Image.open(buf)
print("原始尺寸：", im.size)
im.thumbnail((120, 120))                  # 等比缩到最长边不超过 120 像素
print("缩略图尺寸：", im.size)

## 练习

以下三题由浅到深，皆为复合型但简短，数据一律用 numpy / list 自造（仿真即可），完成后请运行确认输出。

In [ ]:
# TODO 1 ·（简单）建筑群立体散点上色（集成：3D 绘图 + 色彩映射）
#   情境：把一小批建筑用平面位置与楼高画成立体散点，并依容积率（FAR, floor area ratio）上色。
#   要求：
#     1. 用 numpy 自造 30 栋建筑的 x、y（平面座标）、height（楼高，当 z 轴）与 far（容积率，仿真数字即可）。
#     2. 用 add_subplot(projection="3d") 开立体轴，画 3D 散点，把 x, y, height 当座标。
#     3. 把 c=far 配上连续 colormap（如 viridis）为每个点上色，并加一条色条。
#   验收：一张立体散点图，较高的点分布在上方，颜色由低 FAR 到高 FAR 连续变化。
# TODO: 在这里完成

In [ ]:
# TODO 2 ·（中间）多视角建筑面板共用色条（集成：3D 绘图 + 多子图排版 + 循环批量填图 + 色条共用）
#   情境：把同一批街廓（block）建筑用三个不同视角各画一格，做成一页三面板报告，三格共用同一色阶。
#   要求：
#     1. 用 numpy 自造一批建筑的 x、y、height 与 gfa（楼地板面积 gross floor area）。
#     2. 用 subplots 搭配 subplot_kw={"projection":"3d"} 开 1x3 立体子图，用循环把同一份数据填进三格。
#     3. 在循环中对每格调用 view_init 给不同仰角 / 方位角，并用同一个 Normalize（依 gfa 全域最小最大值）上色一致。
#     4. 用 ScalarMappable 搭配该 Normalize 为整张图加一条共用色条。
#   验收：三个视角的立体散点并排，三格颜色对应同一数值范围，右侧一条共用色条同时服务三格。
# TODO: 在这里完成

In [ ]:
# TODO 3 ·（稍难）政策情境前后的网格容积聚合对比
#   （集成：多子图排版 + 轴共享 + 离散色彩 BoundaryNorm+ListedColormap + 色条 + base64 内嵌）
#   情境：把建筑依平面位置聚合到方格（grid cell），比较「现况」与「套高度乘数后」两情境各网格的总楼地板面积分级，
#         并把结果图内嵌成 HTML 字符串。
#   要求：
#     1. 用 numpy 自造建筑的 x、y、footprint（占地面积）与 floors（楼层数），令 gfa = footprint * floors。
#     2. 把 x, y 切成 NxN 网格，聚合落在同一格的建筑 gfa 加总得现况容积矩阵；对 floors 乘高度乘数重算得政策后矩阵。
#     3. 用 subplots(1, 2, sharex=True, sharey=True) 并列两矩阵，用同一组 BoundaryNorm + ListedColormap 做离散分级
#        （低 / 中 / 高 / 极高四级），两图共用一条离散色条。
#     4. 用 savefig(BytesIO) 把整张图存进内存字节流，编 base64 组成 <img> 字符串并印出开头片段。
#   验收：左右两格网格热区并排、轴刻度共享、同一套离散色阶分级，套乘数后高等级网格明显变多，且印出 base64 <img> 开头。
# TODO: 在这里完成

## 小结

- 3D 绘图用 `add_subplot(projection="3d")` 取得立体轴，多了 z 维度与 `view_init` 视角控制。
- 颜色的本质是「数值经范式后查色票」；连续用 `Normalize`，离散分级用 `BoundaryNorm` + `ListedColormap`。
- 多图比较必须共用同一 `Normalize`，并用 `ScalarMappable` 画出一条共用色条，否则同色不同义会误导。
- 多子图用 `subplots` 开网格、循环批量填图，`sharex` / `sharey` 共享刻度让对比更清楚。
- 动画是逐格更新同一张图，`FuncAnimation` 配更新函数，`PillowWriter` 输出 GIF。
- 图可存进 `BytesIO` 不落地，编 base64 内嵌 HTML，再用 PIL `thumbnail` 控制大小。